# Chimera — Kaggle vLLM multimodal server (offline)

Notebook này phục vụ **Qwen3-VL-30B-A3B-Instruct** qua OpenAI-compatible API để Chimera dùng text, ảnh/screenshot và tool calling.

Trước khi chạy:

1. Chọn accelerator **GPU**. Cấu hình mục tiêu là một GPU NVIDIA 96 GB.
2. Gắn một trong hai model vào `/kaggle/input`:
   - `Qwen/Qwen3-VL-30B-A3B-Instruct-FP8` — ưu tiên cho nhiều agent.
   - `Qwen/Qwen3-VL-30B-A3B-Instruct` — BF16, chất lượng gốc nhưng ít VRAM trống hơn.
3. Gắn Kaggle Dataset chứa wheelhouse vLLM offline, tương thích Python/CUDA của image Kaggle hiện tại.
4. Chạy các cell theo thứ tự. Notebook không tải package/model từ Internet.

> `127.0.0.1:8000` chỉ nằm trong Kaggle runtime. Nếu terminal chạy qua Kaggle Jupyter Server thì dùng trực tiếp URL này. Process thực sự chạy trên máy cá nhân cần tunnel/proxy có xác thực; kết nối Jupyter không tự động public cổng 8000.

In [ ]:
# 1) Kiểm tra GPU, CUDA và các input đã gắn
import subprocess
import torch
from pathlib import Path

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
    check=True, capture_output=True, text=True
).stdout)
print(f"PyTorch={torch.__version__}; torch CUDA={torch.version.cuda}")
print(f"GPU={torch.cuda.get_device_name(0)}; compute capability={torch.cuda.get_device_capability(0)}")
nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
print(nvcc.stdout if nvcc.returncode == 0 else "nvcc không có trong PATH")
configs = list(Path("/kaggle/input").rglob("config.json"))
print(f"Tìm thấy {len(configs)} file config.json dưới /kaggle/input")
for path in configs:
    print(path)

In [ ]:
%%bash
# 2) Cài vLLM hoàn toàn từ wheelhouse đã attach
set -euo pipefail

WHEELHOUSE_DIR="$(find /kaggle/input -maxdepth 8 -type d -name wheelhouse -print -quit)"
if [ -z "${WHEELHOUSE_DIR}" ]; then
  VLLM_DIST="$(find /kaggle/input -maxdepth 10 -type f \
    \( -iname 'vllm*.whl' -o -iname 'vllm*.tar.gz' \) -print -quit)"
  if [ -n "${VLLM_DIST}" ]; then
    WHEELHOUSE_DIR="$(dirname "${VLLM_DIST}")"
  fi
fi
if [ -z "${WHEELHOUSE_DIR}" ]; then
  echo "Không tìm thấy wheelhouse vLLM trong /kaggle/input." >&2
  echo "Hãy attach dataset wheelhouse tương thích rồi chạy lại cell." >&2
  exit 1
fi
echo "WHEELHOUSE_DIR=${WHEELHOUSE_DIR}"
python -m pip install --no-index --find-links "${WHEELHOUSE_DIR}" vllm
python -c 'import vllm; print("vLLM", vllm.__version__)'
command -v vllm

In [ ]:
# 3) Chọn model offline và ghi cấu hình server
import json
import os
from pathlib import Path

# Có thể điền đường dẫn tuyệt đối để bỏ qua auto-discovery.
PREFERRED_MODEL_PATH = ""
SERVED_MODEL_NAME = "Qwen/Qwen3-VL-30B-A3B-Instruct"
API_KEY = os.environ.get(
    "CHIMERA_FOUNDATION_API_KEY",
    os.environ.get("CHIMERA_VLLM_API_KEY", "chimera-local-change-me"),
)

def inspect_model(config_path: Path):
    try:
        config = json.loads(config_path.read_text())
    except Exception:
        return None
    model_type = config.get("model_type", "")
    text_config = config.get("text_config", {})
    path_lower = str(config_path.parent).lower()
    is_qwen3_vl_moe = model_type == "qwen3_vl_moe"
    is_30b_a3b = (
        "30b-a3b" in path_lower
        or (text_config.get("num_experts", 0) >= 64 and text_config.get("num_hidden_layers", 0) >= 40)
    )
    is_instruct = "thinking" not in path_lower
    if not (is_qwen3_vl_moe and is_30b_a3b and is_instruct):
        return None
    quant = config.get("quantization_config") or {}
    quant_method = str(quant.get("quant_method", "")).lower()
    score = 100 + (20 if quant_method == "fp8" or "fp8" in path_lower else 0)
    return score, config_path.parent, quant_method or "bf16/auto"

if PREFERRED_MODEL_PATH:
    model_path = Path(PREFERRED_MODEL_PATH)
    if not (model_path / "config.json").is_file():
        raise FileNotFoundError(f"MODEL_PATH không hợp lệ: {model_path}")
    selected_quant = "model config"
else:
    candidates = []
    for config_path in Path("/kaggle/input").rglob("config.json"):
        candidate = inspect_model(config_path)
        if candidate:
            candidates.append(candidate)
    if not candidates:
        raise FileNotFoundError(
            "Không tìm thấy Qwen3-VL-30B-A3B-Instruct dưới /kaggle/input. "
            "Hãy attach bản FP8 hoặc BF16; không attach bản Thinking."
        )
    _, model_path, selected_quant = sorted(candidates, key=lambda item: item[0], reverse=True)[0]

server_config = {
    "model_path": str(model_path),
    "served_model_name": SERVED_MODEL_NAME,
    "api_key": API_KEY,
    "host": "0.0.0.0",
    "port": 8000,
    "max_model_len": 32768,
    "gpu_memory_utilization": 0.90,
    # RTX PRO 6000 Blackwell là SM120. Ép Triton để tránh FlashInfer
    # CUTLASS MoE JIT khi wheelhouse/CUDA toolkit chưa hỗ trợ SM120.
    "moe_backend": "triton",
    "max_num_seqs": 4,
    "max_num_batched_tokens": 8192
}
Path("/kaggle/working/chimera-vllm-config.json").write_text(json.dumps(server_config, indent=2))
print(f"MODEL_PATH={model_path}")
print(f"QUANTIZATION={selected_quant}")
print(f"SERVED_MODEL={SERVED_MODEL_NAME}")
print(f"MOE_BACKEND={server_config['moe_backend']}")
print("Đã ghi /kaggle/working/chimera-vllm-config.json")

In [ ]:
# 4) Dừng server cũ (nếu có) rồi khởi chạy vLLM ở background
import json
import os
import signal
import subprocess
import time
from pathlib import Path

pid_file = Path("/kaggle/working/vllm.pid")
if pid_file.exists():
    try:
        old_pid = int(pid_file.read_text().strip())
        os.killpg(old_pid, signal.SIGTERM)
        time.sleep(4)
        print(f"Đã dừng process group cũ: {old_pid}")
    except (ValueError, ProcessLookupError, PermissionError):
        pass

config = json.loads(Path("/kaggle/working/chimera-vllm-config.json").read_text())
if config.get("moe_backend") != "triton":
    raise RuntimeError(
        "Cấu hình cũ hoặc chưa chạy lại cell 3: moe_backend phải là 'triton'."
    )
# vLLM mới chỉ hiện engine options trong --help=all; --help thường chỉ
# liệt kê các nhóm cấu hình nên không được dùng để kết luận option thiếu.
help_commands = [
    ["vllm", "serve", "--help=moe-backend"],
    ["vllm", "serve", "--help=all"],
    ["vllm", "serve", "--help"],
]
serve_help_text = ""
for help_command in help_commands:
    help_result = subprocess.run(help_command, capture_output=True, text=True)
    candidate_help = help_result.stdout + "\n" + help_result.stderr
    serve_help_text += "\n" + candidate_help
    if "--moe-backend" in candidate_help:
        break
supports_moe_backend_cli = "--moe-backend" in serve_help_text
if supports_moe_backend_cli:
    print("[INFO] vLLM hỗ trợ CLI --moe-backend; sẽ ép Triton.")
else:
    print(
        "[WARN] vLLM chưa có CLI --moe-backend; sẽ dùng biến môi trường "
        "legacy để tắt FlashInfer MoE FP16/BF16."
    )
config["moe_strategy"] = (
    "cli: --moe-backend triton"
    if supports_moe_backend_cli
    else "legacy env: VLLM_USE_FLASHINFER_MOE_FP16=0"
)
Path("/kaggle/working/chimera-vllm-config.json").write_text(
    json.dumps(config, indent=2)
)
log_path = Path("/kaggle/working/vllm.log")
if log_path.exists():
    log_path.rename(log_path.with_name(f"vllm.log.{time.strftime('%H%M%S')}"))
cmd = [
    "vllm", "serve", config["model_path"],
    "--host", config["host"],
    "--port", str(config["port"]),
    "--served-model-name", config["served_model_name"],
    "--dtype", "auto",
    "--max-model-len", str(config["max_model_len"]),
    "--gpu-memory-utilization", str(config["gpu_memory_utilization"]),
]
if supports_moe_backend_cli:
    cmd += ["--moe-backend", config["moe_backend"]]
cmd += [
    "--max-num-seqs", str(config["max_num_seqs"]),
    "--max-num-batched-tokens", str(config["max_num_batched_tokens"]),
    "--limit-mm-per-prompt", '{"image": 2, "video": 0}',
    "--generation-config", "vllm",
    "--enable-prefix-caching",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "hermes",
    "--api-key", config["api_key"]
]
env = os.environ.copy()
env.update({
    "HF_HOME": "/kaggle/working/hf-cache",
    "TRANSFORMERS_CACHE": "/kaggle/working/hf-cache",
    "VLLM_WORKER_MULTIPROC_METHOD": "spawn",
    "VLLM_USE_FLASHINFER_SAMPLER": "0",
    # Giữ các cờ này ở 0 ngay cả khi đã dùng CLI để ngăn những đường
    # FlashInfer legacy được bật bởi environment của Kaggle.
    "VLLM_USE_FLASHINFER_MOE_FP16": "0",
    "VLLM_USE_FLASHINFER_MOE_FP8": "0",
    "VLLM_USE_FLASHINFER_MOE_FP4": "0"
})
log_handle = log_path.open("w")
process = subprocess.Popen(
    cmd, stdout=log_handle, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
    env=env, start_new_session=True
)
log_handle.close()
pid_file.write_text(str(process.pid))
print("Lệnh:", " ".join(cmd[:-2] + ["--api-key", "***"]))
print(
    "MoE strategy:",
    config["moe_strategy"],
)
print(f"vLLM PID={process.pid}; log={log_path}")

In [ ]:
# 5) Xem log vLLM thủ công; bấm Run lại cell này mỗi khi muốn refresh
import json
import urllib.request
from datetime import datetime
from pathlib import Path
from IPython.display import clear_output

TAIL_LINES = 120
MAX_TAIL_BYTES = 512 * 1024
config_path = Path("/kaggle/working/chimera-vllm-config.json")
pid_path = Path("/kaggle/working/vllm.pid")
log_path = Path("/kaggle/working/vllm.log")

def read_log_tail(path: Path, line_count: int) -> str:
    if not path.exists():
        return "(vllm.log chưa được tạo)"
    with path.open("rb") as log_file:
        log_file.seek(0, 2)
        size = log_file.tell()
        start = max(0, size - MAX_TAIL_BYTES)
        log_file.seek(start)
        text = log_file.read().decode("utf-8", errors="replace")
    lines = text.splitlines()
    if start > 0 and lines:
        lines = lines[1:]
    return "\n".join(lines[-line_count:]) or "(log đang trống)"

def read_process_state(pid):
    if not pid:
        return None
    try:
        # Trường đầu sau dấu ')' cuối cùng là state: R/S/D/Z/X...
        stat_text = Path(f"/proc/{pid}/stat").read_text()
        return stat_text.rsplit(")", 1)[1].strip().split()[0]
    except (FileNotFoundError, IndexError):
        return None

config = json.loads(config_path.read_text()) if config_path.exists() else {}
server_pid = int(pid_path.read_text().strip()) if pid_path.exists() else None
process_state = read_process_state(server_pid)
alive = process_state not in (None, "Z", "X")
ready = False
api_error = None
if alive and config:
    models_url = f"http://127.0.0.1:{config['port']}/v1/models"
    request = urllib.request.Request(
        models_url,
        headers={"Authorization": f"Bearer {config['api_key']}"},
    )
    try:
        with urllib.request.urlopen(request, timeout=1) as response:
            ready = response.status == 200
    except Exception as exc:
        api_error = str(exc)

log_tail = read_log_tail(log_path, TAIL_LINES)
engine_failed = "Engine core initialization failed" in log_tail
state = (
    "READY" if ready else ("FAILED" if engine_failed else ("RUNNING" if alive else "EXITED"))
)
clear_output(wait=True)
print(
    f"[{datetime.now().strftime('%H:%M:%S')}] PID={server_pid or 'N/A'} "
    f"STATE={state} PROC={process_state or 'N/A'} MOE={config.get('moe_backend', 'N/A')}"
)
if config:
    print(f"Model: {config.get('served_model_name')}")
    print(f"Endpoint: http://127.0.0.1:{config.get('port')}/v1")
    print(f"MoE strategy: {config.get('moe_strategy', 'chưa ghi; hãy chạy lại cell 4')}")
if api_error:
    print(f"API: chưa sẵn sàng ({api_error})")
print(f"\n--- vllm.log ({TAIL_LINES} dòng cuối) ---")
print(log_tail)
print("\nBấm Run lại cell này để cập nhật log.")

In [ ]:
# 6) Chờ server sẵn sàng (lần đầu load model có thể mất vài phút)
import json
import time
import urllib.request
from pathlib import Path

config = json.loads(Path("/kaggle/working/chimera-vllm-config.json").read_text())
pid_file = Path("/kaggle/working/vllm.pid")
server_pid = int(pid_file.read_text().strip())
models_url = f"http://127.0.0.1:{config['port']}/v1/models"
deadline = time.time() + 900
last_error = None
ready = False

def process_is_running(pid):
    try:
        stat_text = Path(f"/proc/{pid}/stat").read_text()
        state = stat_text.rsplit(")", 1)[1].strip().split()[0]
        return state not in ("Z", "X")
    except (FileNotFoundError, IndexError):
        return False

while time.time() < deadline:
    if not process_is_running(server_pid):
        last_error = RuntimeError(f"vLLM process {server_pid} đã thoát")
        break
    current_log = Path("/kaggle/working/vllm.log").read_text(errors="replace")
    if "Engine core initialization failed" in current_log[-262144:]:
        last_error = RuntimeError("vLLM EngineCore đã khởi tạo thất bại")
        break
    request = urllib.request.Request(models_url, headers={"Authorization": f"Bearer {config['api_key']}"})
    try:
        with urllib.request.urlopen(request, timeout=5) as response:
            print(response.read().decode())
            print("vLLM đã sẵn sàng.")
            ready = True
            break
    except Exception as exc:
        last_error = exc
        print(".", end="", flush=True)
        time.sleep(5)
if not ready:
    log_tail = "\n".join(Path("/kaggle/working/vllm.log").read_text(errors="replace").splitlines()[-120:])
    raise RuntimeError(f"vLLM không sẵn sàng: {last_error}\n--- log tail ---\n{log_tail}")

In [ ]:
# 7) Smoke test text và structured tool calling
import json
import urllib.request
from pathlib import Path

config = json.loads(Path("/kaggle/working/chimera-vllm-config.json").read_text())
endpoint = f"http://127.0.0.1:{config['port']}/v1/chat/completions"
payload = {
    "model": config["served_model_name"],
    "messages": [{"role": "user", "content": "Hãy gọi tool search_web để tìm tài liệu chính thức về Sysdig. Không tự trả lời."}],
    "tools": [{
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Tìm kiếm thông tin trên web",
            "parameters": {
                "type": "object",
                "properties": {"query": {"type": "string"}},
                "required": ["query"]
            }
        }
    }],
    "tool_choice": "auto",
    "temperature": 0,
    "max_tokens": 256
}
request = urllib.request.Request(
    endpoint, data=json.dumps(payload).encode(), method="POST",
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {config['api_key']}"}
)
with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read())
print(json.dumps(result, indent=2, ensure_ascii=False))
tool_calls = result["choices"][0]["message"].get("tool_calls")
assert tool_calls, "Model không trả về structured tool_calls. Kiểm tra tool parser/chat template."
assert tool_calls[0]["function"]["name"] == "search_web"
print("Tool-calling smoke test: OK")

In [ ]:
# 8) Smoke test vision/OCR bằng ảnh tự tạo, không cần Internet
import base64
import io
import json
import urllib.request
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display

image = Image.new("RGB", (900, 220), "white")
draw = ImageDraw.Draw(image)
draw.rectangle((20, 20, 880, 200), outline="navy", width=5)
draw.text((70, 85), "CHIMERA VISION CHECK: 7429", fill="black", font_size=42)
buffer = io.BytesIO()
image.save(buffer, format="PNG")
image_url = "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode()
display(image)
config = json.loads(Path("/kaggle/working/chimera-vllm-config.json").read_text())
payload = {
    "model": config["served_model_name"],
    "messages": [{
        "role": "user",
        "content": [
            {"type": "text", "text": "Đọc chính xác dòng chữ và mã số trong ảnh."},
            {"type": "image_url", "image_url": {"url": image_url}}
        ]
    }],
    "temperature": 0,
    "max_tokens": 128
}
request = urllib.request.Request(
    f"http://127.0.0.1:{config['port']}/v1/chat/completions",
    data=json.dumps(payload).encode(), method="POST",
    headers={"Content-Type": "application/json", "Authorization": f"Bearer {config['api_key']}"}
)
with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read())
answer = result["choices"][0]["message"]["content"]
print(answer)
assert "7429" in answer, "Vision/OCR test không đọc đúng mã 7429."
print("Vision/OCR smoke test: OK")

## Kết nối Chimera

Nếu Chimera được chạy trong terminal/kernel của cùng Kaggle Jupyter Server:

```bash
export OPENAI_BASE_URL=http://127.0.0.1:8000/v1
export OPENAI_API_KEY=chimera-local-change-me
```

Tên model gửi trong request là `Qwen/Qwen3-VL-30B-A3B-Instruct`.

Nếu Chimera chạy như process trên máy cá nhân, `127.0.0.1` của máy cá nhân không phải Kaggle. Khi đó cần port forwarding hoặc HTTPS tunnel có authentication tới cổng 8000. Không dùng API key mặc định khi endpoint có thể truy cập từ Internet.